# Trace GPT-2 **decode** — turn an inference into a measurement

### What this is for

`dynshape` derives kernel shapes for any `(batch, seq_len, mode)` from three traced anchors. For
**prefill** that is verified: it reproduces all 25 shipped GPT-2 templates exactly.

For **decode** it is not, because **no decode workload ships with the EnergAIzer artifact** — all 90
files are `modeprefill`, even though the authors' own `run_gpt2.sh` has `MODE="prefill decode"`. So
decode currently rests on two unchecked assumptions:

1. decode runs the **same 242 kernels in the same order** as prefill
2. the sequence exponent splits into query and key axes by a **hand-written rule**

This notebook replaces both with measurements:

| § | question | how |
|---|---|---|
| 8 | which kernels actually run, and in what order? | read the trace |
| 9 | are the inferred shapes right? | inferred vs measured, entry by entry |
| 10 | does a decode law generalise? | learn from 3 anchors, test on a **4th held-out** shape |

### Runtime

Tracing records **which ops ran and their tensor shapes**, not timings — so no A100 is needed. A free
Colab **T4** is the safe choice. CPU also works but exercises less-tested bf16 kernels, and the
control in §6 will tell you if that became a problem.

`Runtime -> Change runtime type -> T4 GPU`

## 1 — Clone the artifact and install dependencies

**We do not pin `torch`.** The artifact's `conda_env.yml` says `torch==2.7.1`, but installing that in
Colab is actively harmful:

- it replaces Colab's CUDA-matched build, often with one that does not match the driver;
- it breaks `torchvision`, which `run_model.py` imports at module level (line 49) even though we only
  need GPT-2 — so the import would fail before any tracing happens;
- torch cannot be swapped inside a live session anyway; it needs a runtime restart.

Colab's own torch is 2.x and has everything `run_model.py` uses. We pin only `transformers`, because
that is what decides which code path GPT-2 takes — the thing we are actually measuring.

In [1]:
import os, subprocess, sys

EN_ROOT = "/content/single_kernel_GPU_model"
PKG     = os.path.join(EN_ROOT, "energaizer-ispass26-artifact-main")
CODE    = os.path.join(PKG, "test", "code")
DYN     = "/content/dynamic_shape_power_sim"

for url, dest in [
    ("https://github.com/shubhamOjha1000/single_kernel_GPU_model.git", EN_ROOT),
    ("https://github.com/shubhamOjha1000/dynamic_shape_power_sim.git", DYN),
]:
    if not os.path.isdir(dest):
        subprocess.run(["git", "clone", "--depth", "1", url, dest], check=True)

# transformers is pinned (it decides GPT-2's code path); torch is NOT -- see above.
#
# torchlens is the OTHER version-sensitive dependency: parse_trace.py reads
# specific column names out of `ModelHistory.to_pandas()`, and those have moved
# between releases. The artifact's conda_env.yml leaves it unpinned, so today's
# PyPI default may not match what the authors used. Section 3 probes the schema
# and tells you exactly what to put here if the default does not work.
TORCHLENS_SPEC = "torchlens"          # e.g. "torchlens==0.1.21" to pin

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers==4.51.3", TORCHLENS_SPEC,
                "pandas", "numpy", "pyyaml"],
               check=True)

for p in (CODE, DYN):
    if p not in sys.path:
        sys.path.insert(0, p)

print("cloned and installed. Run the next cell to check the environment.")

cloned and installed. Run the next cell to check the environment.


## 2 — Preflight

Import everything **before** tracing anything, so a broken environment fails here with a clear
message rather than deep inside a trace.

If it says the transformers version is wrong, do `Runtime -> Restart session` and re-run from §1 —
a version already imported in this session cannot be swapped underneath it.

In [2]:
import torch, torchvision, transformers, torchlens, pandas, numpy

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch        {torch.__version__}")
print(f"torchvision  {torchvision.__version__}")
print(f"transformers {transformers.__version__}")
print(f"device       {DEVICE}"
      + (f"  ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else ""))

WANT = "4.51.3"
if transformers.__version__ != WANT:
    raise RuntimeError(
        f"transformers is {transformers.__version__}, need {WANT}.\n"
        "Runtime -> Restart session, then re-run from section 1.\n"
        "(The version matters: it decides which branch GPT-2 takes for a KV cache,"
        " which is exactly what we are measuring.)")

# The imports that actually matter -- these are what break first if anything is off.
from run_model import get_model, get_input, run_torchlens   # needs torch, torchvision, torchlens
from parse_trace import parse                               # needs pandas, yaml
from dynshape import ShapeRewriter, learn_scaling, rewrite_dims

print("\nall imports OK")

torch        2.11.0+cu128
torchvision  0.26.0+cu128
transformers 4.51.3
device       cuda  (Tesla T4)

all imports OK


## 3 — The workload config

`run_model.py` needs a config JSON and `workload_config/` **does not ship** with the artifact. We
reconstruct it from the shipped templates:

| evidence in `..._b8_s128_modeprefill.json` | implies |
|---|---|
| `layernorm dim = 768` | `n_embd = 768` |
| `attn batch = 96` at b=8 | `n_head = 12` |
| the block repeats 12 times | `n_layer = 12` |
| QKV `dimN = 2304` = 3 × 768 | standard fused QKV |
| MLP `dimN = 3072` = 4 × 768 | standard |

That is GPT-2 base. One value must be overridden: `n_positions` defaults to **1024**, but `s4096`
templates exist, so the authors raised it too. It does not affect the traced kernels — the embedding
lookup is dropped by the tracer's keep-list — but the model will not build without it.

In [3]:
import json

CONFIG_DIR = os.path.join(CODE, "workload_config", "gpt2")
os.makedirs(CONFIG_DIR, exist_ok=True)
CONFIG_FILE = os.path.join(CONFIG_DIR, "gpt2.json")   # stem "gpt2" -> matches shipped filenames

with open(CONFIG_FILE, "w") as f:
    json.dump({"n_embd": 768, "n_head": 12, "n_layer": 12,
               "vocab_size": 50257, "n_positions": 8192}, f, indent=1)

print(open(CONFIG_FILE).read())

{
 "n_embd": 768,
 "n_head": 12,
 "n_layer": 12,
 "vocab_size": 50257,
 "n_positions": 8192
}


## 4 — The tracing helper

We call `get_model` / `get_input` / `run_torchlens` **directly** rather than through
`run_model.py`'s CLI, because that path is unusable here:

- it calls `get_time_per_iter()` before tracing, which calls `torch.cuda.synchronize()` — that fails
  outright on CPU;
- its whole body sits inside `except Exception as e: print(e)`, so any failure prints one line and
  silently produces **no trace at all**.

Same tracing function, no swallowed errors. We round-trip through CSV exactly as the real pipeline
does, so `parse_trace` sees identical input.

In [4]:
import gc
import pandas as pd

MODEL_TYPE = ("LanguageModel", "GPT2Model")
DTYPE      = torch.bfloat16
OUT        = "/content/traces"
os.makedirs(OUT, exist_ok=True)


def norm(entries):
    """Canonical form for comparison.

    `parse()` returns (dict, tuple) but `json.load()` returns [dict, list], so a
    raw `==` between a fresh trace and a shipped template is ALWAYS False. Without
    this the control below would fail for a reason unrelated to the trace.
    """
    return [(dict(q), tuple(op)) for q, op in entries]


# --- torchlens schema adapter ----------------------------------------------
# parse_trace.py reads these nine columns out of ModelHistory.to_pandas().
# (`fusion`, `fusion_ignore_*` and `fused_op_count` are created by parse itself;
# `conv_stride` / `conv_padding` are added by run_torchlens for vision models,
# which GPT-2 never reaches.)
REQUIRED = ["layer_label", "layer_type", "parent_layers", "parent_param_shapes",
            "tensor_shape", "tensor_dtype", "containing_module_origin",
            "computed_with_params", "num_params_total"]

# Names torchlens has used for the same field across releases. If your version
# uses something not listed here, the probe below prints the real column list.
ALIASES = {
    "layer_label":              ["layer_label_w_pass", "label", "layer_labels"],
    "layer_type":               ["func_applied_name", "layer_type_str", "op_type"],
    "parent_layers":            ["parents", "parent_layer_labels", "parent_layer_names",
                                 "parent_layers_labels", "parent_layer_label"],
    "parent_param_shapes":      ["parent_params_shape", "param_shapes",
                                 "parent_param_shape", "parent_params_shapes"],
    "tensor_shape":             ["shape", "tensor_shapes"],
    "tensor_dtype":             ["dtype", "tensor_dtypes"],
    "containing_module_origin": ["containing_modules_origin_nested", "module_origin",
                                 "containing_module"],
    "computed_with_params":     ["has_params", "computed_from_params", "computed_with_param"],
    "num_params_total":         ["num_param_tensors", "total_params", "num_params"],
}


def adapt_columns(df):
    """Rename torchlens output to the schema parse_trace expects.

    Returns (df, renamed, missing). A non-empty `missing` means this torchlens
    version emits a field under a name not in ALIASES -- the probe prints the
    real columns so the mapping can be extended.
    """
    df = df.copy()
    renamed, missing = {}, []
    for canon in REQUIRED:
        if canon in df.columns:
            continue
        hit = next((a for a in ALIASES.get(canon, []) if a in df.columns), None)
        if hit is None:
            missing.append(canon)
        else:
            df[canon] = df[hit]
            renamed[canon] = hit
    return df, renamed, missing


def trace_shape(batch, seqlen, mode, prec="bf16"):
    """Trace one (batch, seqlen, mode); returns the parsed kernel entries."""
    name      = f"gpt2model_gpt2_p{prec}_b{batch}_s{seqlen}_mode{mode}"
    csv_path  = os.path.join(OUT, name + ".csv")
    json_path = os.path.join(OUT, name + ".json")

    module, config = get_model(MODEL_TYPE, CONFIG_FILE, DEVICE, DTYPE, "eager")
    inp = get_input(MODEL_TYPE, config, batch, seqlen, mode, DTYPE, DEVICE)

    try:
        df = run_torchlens(module, MODEL_TYPE, inp["input"], inp["past_kv"],
                           inp["use_cache"], seqlen)
    except Exception as e:
        if mode != "decode":
            raise
        # get_input() builds the KV cache as a legacy list-of-[k, v]; transformers
        # 4.51 may refuse it. Converting is a compatibility shim, NOT a no-op --
        # it could change which branch the model takes, so it is reported loudly.
        print(f"  [SHIM] legacy list cache rejected ({type(e).__name__}: {e}); "
              f"retrying with DynamicCache -- mention this when reporting results")
        from transformers.cache_utils import DynamicCache
        cache = DynamicCache.from_legacy_cache(
            tuple((kv[0], kv[1]) for kv in inp["past_kv"]))
        df = run_torchlens(module, MODEL_TYPE, inp["input"], cache,
                           inp["use_cache"], seqlen)

    df, renamed, missing = adapt_columns(df)
    if missing:
        raise KeyError(
            f"torchlens produced no column for {missing}. Run the schema probe in "
            f"section 5 and extend ALIASES, or pin TORCHLENS_SPEC in section 1.")
    if renamed:
        print(f"  [schema] mapped {renamed}")

    df.to_csv(csv_path, index=False)
    entries = norm(parse(pd.read_csv(csv_path), False))
    with open(json_path, "w") as f:
        json.dump([[q, list(op)] for q, op in entries], f)

    del module, inp
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    print(f"  {name}: {len(entries)} kernels")
    return entries


print("helper ready, device:", DEVICE, "| dtype:", DTYPE)

helper ready, device: cuda | dtype: torch.bfloat16


## 5 — Schema probe: does this torchlens version match `parse_trace`?

**This is the cheapest failure to find.** `parse_trace.py` reads nine specific column names out of
`ModelHistory.to_pandas()`, and torchlens has moved those names between releases. The artifact leaves
torchlens unpinned, so today's PyPI default need not match what the authors used.

A tiny trace (`b1, s8`) answers it in seconds instead of failing partway through a full one — the
original symptom was a `KeyError: \'parent_layers\'` raised from deep inside `parse()`.

In [5]:
_m, _c = get_model(MODEL_TYPE, CONFIG_FILE, DEVICE, DTYPE, "eager")
_i = get_input(MODEL_TYPE, _c, 1, 8, "prefill", DTYPE, DEVICE)
probe_df = run_torchlens(_m, MODEL_TYPE, _i["input"], _i["past_kv"], _i["use_cache"], 8)
del _m, _i; gc.collect()

print(f"torchlens {torchlens.__version__} produced {len(probe_df.columns)} columns:\n")
cols = sorted(probe_df.columns)
for i in range(0, len(cols), 3):
    print("   " + "".join(f"{c:<34}" for c in cols[i:i + 3]))

_, renamed, missing = adapt_columns(probe_df)

print(f"\n{'required column':<28}{'status'}")
print("-" * 60)
for c in REQUIRED:
    if c in probe_df.columns:
        print(f"{c:<28}present")
    elif c in renamed:
        print(f"{c:<28}mapped from {renamed[c]!r}")
    else:
        print(f"{c:<28}MISSING")

if missing:
    raise RuntimeError(
        f"\ntorchlens {torchlens.__version__} has no column for {missing}.\n\n"
        "Two ways forward:\n"
        "  1. Look through the column list printed above for the field under a\n"
        "     different name, add it to ALIASES in section 4, and re-run.\n"
        "  2. Pin an older release: set TORCHLENS_SPEC in section 1 (e.g.\n"
        "     \"torchlens==0.1.21\"), then Runtime -> Restart session and re-run.\n\n"
        "Either way, the column list above is the information needed -- paste it "
        "when reporting this.")

print("\nschema OK -- parse_trace can read this torchlens version.")

torchlens 2.34.1 produced 155 columns:

   activation_memory                 address                           arg_expressions                   
   arg_names                         autograd_memory                   backend_address                   
   bool_value                        buffer_pass                       buffer_replay_validated           
   buffer_source                     buffer_source_func_name           buffer_value_changed              
   buffer_write_kind                 bytes_delta_at_call               bytes_peak_at_call                
   children                          co_parents                        conditional_arm_children          
   conditional_branch_depth          conditional_branch_stack          conditional_context_kind          
   conditional_elif_children         conditional_else_children         conditional_entry_children        
   conditional_then_children         conditional_wrapper_kind          container_path                    
   det

RuntimeError: 
torchlens 2.34.1 has no column for ['containing_module_origin', 'computed_with_params'].

Two ways forward:
  1. Look through the column list printed above for the field under a
     different name, add it to ALIASES in section 4, and re-run.
  2. Pin an older release: set TORCHLENS_SPEC in section 1 (e.g.
     "torchlens==0.1.21"), then Runtime -> Restart session and re-run.

Either way, the column list above is the information needed -- paste it when reporting this.

## 6 — The control: reproduce a template that already exists

**Run this before touching decode.** Trace `(b8, s128, prefill)` and compare against the shipped
`gpt2model_gpt2_pbf16_b8_s128_modeprefill.json`.

If the pipeline reproduces a known file **exactly**, decode traces from that same pipeline can be
trusted. If it does not — wrong config, wrong transformers version, wrong attention backend, a bf16
CPU kernel behaving differently — we find out here, on a case with a known answer, instead of
misreading a decode result later.

The next cell refuses to run if this fails.

In [ ]:
SHIPPED = os.path.join(PKG, "test", "data", "workloads", "all")

ctrl = trace_shape(8, 128, "prefill")
ref  = norm(json.load(open(os.path.join(
           SHIPPED, "gpt2model_gpt2_pbf16_b8_s128_modeprefill.json"))))

print(f"\n  traced : {len(ctrl)} kernels")
print(f"  shipped: {len(ref)} kernels")

CONTROL_OK = (ctrl == ref)

if CONTROL_OK:
    print("\n  EXACT MATCH -- the pipeline reproduces a known template.")
else:
    print("\n  MISMATCH -- do not trust any decode result until this is understood.\n")
    from collections import Counter
    cc, cr = Counter(op for _, op in ctrl), Counter(op for _, op in ref)
    print(f"  {'op':<28}{'traced':>8}{'shipped':>9}")
    for op in sorted(set(cc) | set(cr)):
        mark = "" if cc.get(op, 0) == cr.get(op, 0) else "   <--"
        print(f"  {' '.join(op):<28}{cc.get(op,0):>8}{cr.get(op,0):>9}{mark}")
    if len(ctrl) == len(ref):
        for i, (a, b) in enumerate(zip(ctrl, ref)):
            if a != b:
                print(f"\n  first differing entry, index {i}:")
                print(f"    traced : {a[0]}")
                print(f"    shipped: {b[0]}")
                break
    print("\n  Most likely causes, in order: transformers version differs from 4.51.3;")
    print("  running on CPU where a bf16 kernel decomposes differently; n_positions")
    print("  affecting the position path. Report this output rather than continuing.")

## 7 — Trace decode

Four shapes. Three anchors teach the scaling law; the fourth is **held out** so the law can be tested
against a trace it has never seen — the same proof structure that gives prefill its 25/25.

| shape | role |
|---|---|
| `b8, s128` | base |
| `b16, s128` | batch doubled → recovers the batch exponent |
| `b8, s512` | context ×4 → recovers the context exponent |
| `b16, s512` | **held out** — testing only, never used for learning |

In decode, `seqlen` is the **KV cache length**: one new token per sequence, attending over that many
stored tokens.

In [ ]:
if not CONTROL_OK:
    raise RuntimeError(
        "The control in section 6 did not match the shipped template. Decode traces "
        "produced by the same pipeline would not be interpretable. Fix that first, or "
        "set CONTROL_OK = True deliberately if you have understood the difference.")

decode_traces = {}
for b, s in [(8, 128), (16, 128), (8, 512), (16, 512)]:
    decode_traces[(b, s)] = trace_shape(b, s, "decode")

print("\ndone:", {f"b{b}_s{s}": len(v) for (b, s), v in decode_traces.items()})

## 8 — Question 1: which kernels run, and in what order?

In [ ]:
from collections import Counter

pre, dec = ctrl, decode_traces[(8, 128)]
print(f"prefill : {len(pre)} kernels")
print(f"decode  : {len(dec)} kernels\n")

same_seq = (len(pre) == len(dec)) and [op for _, op in pre] == [op for _, op in dec]

if same_seq:
    print("The op SEQUENCE is identical -- decode runs the same kernels in the same")
    print("order, only the shapes differ. The structural assumption behind the")
    print("inferred rule holds.")
else:
    print("The op sequence DIFFERS. The inferred rule was wrong about STRUCTURE,")
    print("not just exponents. This is the most valuable outcome here: decode needs")
    print("its own template, which these traces now provide.")

print(f"\n{'op':<28}{'prefill':>9}{'decode':>9}")
cp, cd = Counter(op for _, op in pre), Counter(op for _, op in dec)
for op in sorted(set(cp) | set(cd)):
    flag = "" if cp.get(op, 0) == cd.get(op, 0) else "   <-- differs"
    print(f"{' '.join(op):<28}{cp.get(op,0):>9}{cd.get(op,0):>9}{flag}")

## 9 — Question 2: were the inferred shapes right?

Side by side: what `dynshape`'s hand-written query/key split **predicted**, against what the trace
**shows**. This is where the guess gets graded.

In [ ]:
rw = ShapeRewriter.from_dir(os.path.join(DYN, "templates", "gpt2"))
print("decode_source:", rw.decode_source, " (no decode traces installed yet)")

B, S = 8, 128
inferred = norm(rw.expand(B, S, "decode"))
measured = decode_traces[(B, S)]

if len(inferred) != len(measured):
    print(f"\nNo entry-by-entry comparison possible: {len(inferred)} inferred vs "
          f"{len(measured)} measured.\nThat difference IS the finding -- see section 8.")
else:
    diffs = [(i, a, b) for i, (a, b) in enumerate(zip(inferred, measured)) if a != b]
    print(f"\n{len(measured) - len(diffs)} / {len(measured)} entries match exactly")
    if not diffs:
        print("\nThe inferred query/key split was CORRECT. It is now measured, not argued.")
    else:
        print(f"\n{len(diffs)} entries differ. First few:\n")
        for i, a, b in diffs[:6]:
            print(f"  entry {i}  ({' '.join(a[1])})")
            print(f"    inferred: {a[0]}")
            print(f"    measured: {b[0]}\n")

## 10 — Question 3: does a learned decode law generalise?

In [ ]:
base, alt_b  = decode_traces[(8, 128)],  decode_traces[(16, 128)]
alt_s, held  = decode_traces[(8, 512)],  decode_traces[(16, 512)]

try:
    rules = learn_scaling(base, alt_b, alt_s, batch_ratio=2, seq_ratio=4)
    print(f"learned a decode law over {sum(len(r) for r in rules)} numeric fields\n")

    generated = rewrite_dims(base, rules, 8, 128, 16, 512)
    if generated == held:
        print("HELD-OUT TEST PASSED -- b16 s512 derived from the three anchors is")
        print("identical to its own independent trace.")
        print("Decode is now as verified as prefill.")
    else:
        bad = [i for i, (a, b) in enumerate(zip(generated, held)) if a != b]
        print(f"HELD-OUT TEST FAILED at {len(bad)} of {len(held)} entries. First:")
        i = bad[0]
        print(f"  derived: {generated[i][0]}")
        print(f"  traced : {held[i][0]}")
        print("\nDecode is not a pure power law. Worth knowing -- it means decode")
        print("needs interpolation across more traces, not extrapolation from three.")
except ValueError as e:
    print("learn_scaling refused:\n ", e)
    print("\nThat is the loud failure working as designed: some field does not follow")
    print("value = const * B^a * S^b, so no law was silently invented.")

## 11 — Install the traces and download

Copy the three anchors into `templates/gpt2/` and `dynshape` picks them up automatically —
`decode_source` flips from `inferred` to `measured`, and the hand-written split rule is never
consulted again.

The held-out `b16 s512` is deliberately **not** installed: it stays a test, not training data.

In [ ]:
import shutil, glob

TEMPLATES = os.path.join(DYN, "templates", "gpt2")
for b, s in [(8, 128), (16, 128), (8, 512)]:
    shutil.copy(os.path.join(OUT, f"gpt2model_gpt2_pbf16_b{b}_s{s}_modedecode.json"),
                TEMPLATES)

rw2 = ShapeRewriter.from_dir(TEMPLATES)
print("decode_source  :", rw2.decode_source)
print("prefill kernels:", rw2.n_kernels("prefill"))
print("decode  kernels:", rw2.n_kernels("decode"))

shutil.make_archive("/content/gpt2_decode_traces", "zip", OUT)
print("\nfiles:", sorted(os.path.basename(f) for f in glob.glob(OUT + "/*.json")))

try:
    from google.colab import files
    files.download("/content/gpt2_decode_traces.zip")
except Exception as e:
    print("(not in Colab -- take /content/gpt2_decode_traces.zip manually)", e)

---

## What to do with the result

Commit the three `..._modedecode.json` files into `templates/gpt2/` in
[dynamic_shape_power_sim](https://github.com/shubhamOjha1000/dynamic_shape_power_sim). Nothing else
changes — `from_dir` finds them, and every decode number afterwards comes from a measured law.

### Reading the outcomes

| § | outcome | meaning |
|---|---|---|
| **5** | a column is MISSING | torchlens schema mismatch — paste the printed column list |
| **6** | mismatch | stop — nothing below is interpretable |
| **8** | sequence identical | the structural assumption held |
| **8** | sequence differs | the inferred rule was wrong about structure — the most valuable result here |
| **9** | all entries match | the guess was right, and is now a measurement |
| **9** | entries differ | every decode power number so far was wrong; these traces are the fix |
| **10** | held-out passes | decode is as verified as prefill |
| **10** | held-out fails | decode is not a pure power law; it needs more traces and interpolation |

Report the `[schema]` line too if one appeared — it means a column had to be renamed, which is worth
recording beside any result. And report it if `[SHIM]` printed — converting the KV cache to `DynamicCache` could change which
branch the model takes, so a trace captured that way may differ from the authors' original.

### What these traces still cannot fix

The **KV-cache write** stays missing. HuggingFace appends with `torch.cat`, and `cat` is absent from
the tracer's keep-list (`linear, addmm, matmul, bmm, mm, mul, add, ... softmax, layernorm`), so it
produces no entry in any mode.

That is acceptable for a vLLM-targeted simulator, and the size is known: under paged attention the
write is ≈295 KB/step for GPT-2 at batch 8, ctx 2048 — about **0.014%** of a decode step. Under
HuggingFace's `torch.cat` it would be ≈1.2 GB/step, **~56%** — which is why HuggingFace measurements
are not a valid comparison target for decode timing.